# Convolutional Neural Networks
You should build an end-to-end machine learning pipeline using a convolutional neural network model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [convolutional neural network](https://keras.io/examples/vision/mnist_convnet/) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [31]:
!pip install tensorflow

In [38]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# 2. Load the MNIST dataset using Pandas
# Assumes you have "mnist.csv" in a "datasets" folder
data = pd.read_csv("mnist.csv")

data.head()

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:

data = data.drop("id", axis=1)
X = data.drop("class", axis=1).values
y = data["class"].values

#normal
X = X / 255.0

#to have  784 pixels per image
X = X.reshape(-1, 28, 28, 1)

# 6. One-hot encode labels
y = to_categorical(y)

# 7. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Build a CNN
#first 64 and  is filterd
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    #keeps only the strongest (max) value in each area (usually 2×2).
    MaxPooling2D((2, 2)),
    # take2D -->  1D vector.
    Flatten(),
    #128 neurons connect to 3136  values from Flatten()
    Dense(128, activation='relu'),
    #10 neurons, one for each digit (0–9) for example output [0.01, 0.03, 0.05, ..., 0.92] → predicted class = 9
    Dense(10, activation='softmax')
])


model.summary()

#categorical_crossentropy is tell how far is our predictions from   correct answers,
# small one is the best
# example
#True label: [0, 0, 0, 1, 0, 0, 0, 0, 0, 0] (means digit 3)

#Prediction: [0.1, 0.1, 0.05, 0.2, ..., 0.05]

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 10. Train the model
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(" accuracy:", test_acc)

# Classification
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_14 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - accuracy: 0.5073 - loss: 1.6527 - val_accuracy: 0.8687 - val_loss: 0.4611
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.8898 - loss: 0.3576 - val_accuracy: 0.9062 - val_loss: 0.2641
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.9418 - loss: 0.1962 - val_accuracy: 0.9469 - val_loss: 0.1777
Epoch 4/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - accuracy: 0.9572 - loss: 0.1361 - val_accuracy: 0.9531 - val_loss: 0.1527
Epoch 5/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.9743 - loss: 0.0858 - val_accuracy: 0.9719 - val_loss: 0.1191
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9650 - loss: 0.0811
 accuracy: 0.9512500166893005
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        70
           1       0.94      0.97      0.96       100
           2       0.93      0.89      0.91        73
           